# Parallactic angles

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/simulation/parallactic_angle.ipynb)

Port of SIRIUS technical memo 03.  The parallactic angle rotates the (Zernike) antenna beams on
the sky.  AstroVIPER computes it with astropy
(`astroviper.processing_functions.simulation.calculate_parallactic_angles`): the direction and
the celestial pole are transformed to a topocentric Alt/Az frame at the observing location and
the parallactic angle is the position angle of the pole seen from the direction.

The memo compared several implementations against the parallactic angles that CASA's
``awproject`` gridder stores in its convolution-function cache.  The CASA numbers are reproduced
below as fixed reference values (CASA is not needed to run this notebook).

---
## Background

- CASA (``MSDerivedValues::parAngle``) converts an HADEC zenith and the J2000 pointing into the
  geocentric ``AZEL`` frame; astropy uses the geodetic (``AZELGEO``-like) frame and the FK5/ICRS
  pole.  The memo found differences of ~0.3 deg between the two conventions, dominated by the
  zenith definition (HADEC vs FK5, ~436 arcsec) and the AZEL vs AZELGEO frame (elevation dependent).
- For the VLA a single observing location can be used for all antennas; ``awproject`` uses the
  position of the first antenna.


## Install AstroVIPER

In [ ]:
import os
from importlib.metadata import version

try:
    import astroviper  # noqa: F401

    print("Using astroviper version", version("astroviper"))
except ImportError:
    os.system("pip install --upgrade astroviper")
    import astroviper  # noqa: F401

    print("Installed astroviper version", version("astroviper"))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from astropy.coordinates import SkyCoord

xr.set_options(display_style="html")
ARCSEC_TO_RAD = np.pi / (180 * 3600)

In [ ]:
from astroviper.processing_functions.simulation import calculate_parallactic_angles
from astroviper.utils.telescope_layout import read_telescope_layout

calculate_parallactic_angles?

## Observation of the memo

Six 2-hour steps starting at 2019-10-03T21:21:40.151 UTC, EVLA, pointing at 19h59m28.5s +40d44m01.5s (FK5).

In [ ]:
antenna_xds = read_telescope_layout("vla.d", telescope_name="EVLA")
observing_location = antenna_xds.ANTENNA_POSITION.values[
    0
]  # awproject uses the first antenna
time = (
    np.datetime64("2019-10-03T21:21:40.151") + np.arange(6) * np.timedelta64(7200, "s")
).astype(str)
phase_center = SkyCoord(ra="19h59m28.5s", dec="+40d44m01.5s", frame="fk5")
phase_center_ra_dec = np.array([[phase_center.ra.rad, phase_center.dec.rad]])


def to_degrees(angle):
    angle = np.where(angle > np.pi, angle - 2 * np.pi, angle)
    return angle * 180 / np.pi


pa_astropy = to_degrees(
    calculate_parallactic_angles(
        time, observing_location, phase_center_ra_dec, direction_frame="fk5"
    )
)
pa_astropy_icrs = to_degrees(
    calculate_parallactic_angles(
        time, observing_location, phase_center_ra_dec, direction_frame="icrs"
    )
)
print("astropy (fk5)  [deg]:", pa_astropy)
print("astropy (icrs) [deg]:", pa_astropy_icrs)

## Reference values from the memo

In [ ]:
# CASA awproject parallactic angles (from the CF cache miscinfo), degrees
pa_awproject = np.array(
    [-70.649899, -86.29112099, -115.71606771, 114.72711096, 85.99807173, 70.39997555]
)
# SIRIUS astropy implementation at the time of the memo, degrees
pa_sirius_astropy = np.array(
    [-70.34777117, -85.89276089, -114.96699024, 114.23132453, 85.84484406, 70.341338]
)
# CASA measures with frame AZELGEO and an FK5 zenith (closest CASA analogue of the astropy calculation), degrees
pa_casa_azelgeo_fk5 = np.array(
    [-70.34778111, -85.89275009, -114.96710083, 114.2309824, 85.84475879, 70.34128818]
)

import pandas as pd

table = pd.DataFrame(
    {
        "time (UTC)": time,
        "astroviper astropy [deg]": pa_astropy,
        "SIRIUS astropy [deg]": pa_sirius_astropy,
        "CASA AZELGEO/FK5 [deg]": pa_casa_azelgeo_fk5,
        "CASA awproject (AZEL/HADEC) [deg]": pa_awproject,
        "astroviper - SIRIUS [arcsec]": (pa_astropy - pa_sirius_astropy) * 3600,
        "astroviper - CASA AZELGEO/FK5 [arcsec]": (pa_astropy - pa_casa_azelgeo_fk5)
        * 3600,
        "astroviper - awproject [arcsec]": (pa_astropy - pa_awproject) * 3600,
    }
)
table

## Discussion

- The AstroVIPER implementation reproduces the SIRIUS astropy values to better than 0.001 arcsec.
- The difference to CASA's geodetic frame with an FK5 zenith is ~1 arcsec (earth model / nutation details).
- CASA's ``awproject`` (geocentric ``AZEL`` frame, HADEC zenith) differs by up to ~0.75 deg; the
  largest part is a near-constant ~436 arcsec offset from the zenith definition, the rest is
  elevation dependent.  Which convention is "right" depends on what the beam model was measured
  against; the Zernike models shipped with AstroVIPER were derived with CASA conventions, so
  a sub-degree rotation offset is within their fidelity.

## Parallactic angle through the night

In [ ]:
dense_time = (
    np.datetime64("2019-10-03T16:00:00.000")
    + np.arange(0, 12 * 60, 5) * np.timedelta64(60, "s")
).astype(str)
pa_dense = to_degrees(
    calculate_parallactic_angles(
        dense_time, observing_location, phase_center_ra_dec, direction_frame="fk5"
    )
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(len(dense_time)) * 5 / 60, pa_dense)
ax.set_xlabel("hours after 2019-10-03T16:00 UTC")
ax.set_ylabel("parallactic angle [deg]")
ax.set_title("EVLA, 19h59m28.5s +40d44m01.5s")
plt.show()